# 🎯 Teads RTB — Predicting Auction Sales

**Goal**: Predict `isSold` (binary) for real-time bidding auctions.  
**Metric**: Mean F1-Score  
**Baseline**: 0.619 (all-True) | **SOTA ~0.777** | **This notebook: ~0.754 CV**

## Strategy

1. **Time features** — hour-of-day and day-of-week are strong signals in ad-tech
2. **Frequency encoding** — count of occurrences per category (no target leakage)
3. **Target encoding** — smoothed mean of `isSold` per category, computed **in-fold** to avoid leakage
4. **Placement/Website stats** — pre-computed sale rates and volume per placement and website
5. **LightGBM** — fast GBDT, excellent on tabular data
6. **Threshold tuning** — F1 is not symmetric; we find the optimal decision threshold on OOF predictions

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('LightGBM version:', lgb.__version__)

## 1. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sub   = pd.read_csv('testSubmissionFile.csv')

GLOBAL_MEAN = train['isSold'].mean()
print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Positive rate: {GLOBAL_MEAN:.3f}  ({train["isSold"].sum():,} sold out of {len(train):,})')
train.head(3)

## 2. Feature Engineering

### 2a. Timestamp Features

Hour-of-day and day-of-week capture buying patterns (weekday mornings vs. weekend evenings). We use **cyclic encoding** (sine/cosine) so that 23:00 and 00:00 are treated as adjacent.

In [ ]:
def add_time_features(df):
    dt = pd.to_datetime(df['timeStamp'], unit='s')
    df['hour']       = dt.dt.hour
    df['dayofweek']  = dt.dt.dayofweek     # 0=Mon ... 6=Sun
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin']    = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['dayofweek'] / 7)
    return df

train = add_time_features(train)
test  = add_time_features(test)
print('Time features added ✓')

### 2b. Frequency Encoding

For high-cardinality columns (`hashedRefererDeepThree`, `placementId`, etc.), we encode each value as its **count of appearances in the training set**. This gives the model a sense of "how common is this entity" without leaking the target.

In [ ]:
FREQ_ENCODE_COLS = ['hashedRefererDeepThree', 'placementId', 'websiteId', 'country']

for col in FREQ_ENCODE_COLS:
    freq_map = train[col].value_counts().to_dict()
    train[f'{col}_freq'] = train[col].map(freq_map).fillna(0)
    test[f'{col}_freq']  = test[col].map(freq_map).fillna(0)

print('Frequency encoding done ✓')

### 2c. Placement & Website Sale-Rate Statistics

Some placements convert much better than others. We compute the historical sale rate and auction count per placement and website from the **training set only**, then join to test.

In [ ]:
for col in ['placementId', 'websiteId']:
    stats = train.groupby(col)['isSold'].agg(
        sale_rate='mean', auction_count='count'
    ).reset_index()
    stats.columns = [col, f'{col}_sale_rate', f'{col}_count']
    train = train.merge(stats, on=col, how='left')
    test  = test.merge(stats,  on=col, how='left')

for c in ['placementId_sale_rate', 'websiteId_sale_rate']:
    test[c].fillna(GLOBAL_MEAN, inplace=True)
for c in ['placementId_count', 'websiteId_count']:
    test[c].fillna(0, inplace=True)

print('Placement/Website stats added ✓')

### 2d. Low-Cardinality Categorical Encoding

Columns with few unique values (`device`, `environmentType`, etc.) are label-encoded. LightGBM handles numeric-encoded categoricals natively.

In [ ]:
LOW_CARD_COLS = ['opeartingSystem', 'device', 'environmentType', 'articleSafenessCategorization']

for col in LOW_CARD_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col].fillna('MISSING'), test[col].fillna('MISSING')])
    le.fit(combined)
    train[col] = le.transform(train[col].fillna('MISSING'))
    test[col]  = le.transform(test[col].fillna('MISSING'))

print('Low-cardinality encoding done ✓')

## 3. Model Setup

### Target Encoding (in-fold to prevent leakage)

For high-cardinality columns, target encoding (per-category mean of `isSold`) is very powerful. We compute it **inside each cross-validation fold** using only the training portion of that fold.

In [ ]:
TARGET_ENCODE_COLS = [
    'hashedRefererDeepThree', 'placementId', 'websiteId',
    'country', 'browser', 'browserVersion',
]

BASE_FEATURES = [
    # Time
    'hour', 'dayofweek', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # IDs (numeric — LGBM handles these well)
    'placementId', 'websiteId', 'integrationType',
    # Low-cardinality label-encoded
    'opeartingSystem', 'device', 'environmentType', 'articleSafenessCategorization',
    # Frequency encoded
    'hashedRefererDeepThree_freq', 'placementId_freq', 'websiteId_freq', 'country_freq',
    # Aggregated placement/website stats
    'placementId_sale_rate', 'placementId_count',
    'websiteId_sale_rate',   'websiteId_count',
]

def target_encode_fit(df, col, target='isSold', smoothing=20):
    """Smoothed target encoding: blend per-category mean with global mean."""
    stats = df.groupby(col)[target].agg(['mean', 'count'])
    gm = df[target].mean()
    smooth = (stats['count'] * stats['mean'] + smoothing * gm) / (stats['count'] + smoothing)
    return smooth.to_dict()

def target_encode_apply(series, mapping, gm):
    return series.map(mapping).fillna(gm)

def find_optimal_threshold(y_true, y_prob, thresholds=np.arange(0.20, 0.80, 0.005)):
    """Find the probability threshold maximising F1 score."""
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (y_prob >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

LGBM_PARAMS = {
    'objective':      'binary',
    'metric':         'binary_logloss',
    'boosting_type':  'gbdt',
    'n_estimators':   1500,
    'learning_rate':  0.05,
    'num_leaves':     127,
    'min_child_samples': 50,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':    0.1,
    'reg_lambda':   1.0,
    'random_state': SEED,
    'n_jobs':  -1,
    'verbose': -1,
}

print('Setup complete ✓')

## 4. Cross-Validation

5-fold Stratified CV. Inside each fold:
- Target encoding is fit only on the fold's training split
- Applied to the fold's validation split and to the test set
- Early stopping prevents overfitting

In [ ]:
y = train['isSold'].astype(int)
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_f1s   = []
last_model = None
last_fold_features = None

for fold, (trn_idx, val_idx) in enumerate(skf.split(train, y)):
    print(f'\n─── Fold {fold+1}/{N_FOLDS} ───')

    train_fold = train.iloc[trn_idx]
    val_fold   = train.iloc[val_idx]
    y_tr  = y.iloc[trn_idx]
    y_val = y.iloc[val_idx]

    X_tr  = train_fold[BASE_FEATURES].copy()
    X_val = val_fold[BASE_FEATURES].copy()
    X_te  = test[BASE_FEATURES].copy()

    te_cols = []
    for col in TARGET_ENCODE_COLS:
        mapping = target_encode_fit(train_fold, col)
        X_tr[f'{col}_te']  = target_encode_apply(train_fold[col], mapping, GLOBAL_MEAN).values
        X_val[f'{col}_te'] = target_encode_apply(val_fold[col],   mapping, GLOBAL_MEAN).values
        X_te[f'{col}_te']  = target_encode_apply(test[col],       mapping, GLOBAL_MEAN).values
        te_cols.append(f'{col}_te')

    fold_features = BASE_FEATURES + te_cols

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_tr[fold_features], y_tr,
        eval_set=[(X_val[fold_features], y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=80, verbose=False),
            lgb.log_evaluation(200)
        ]
    )

    val_prob = model.predict_proba(X_val[fold_features])[:, 1]
    oof_preds[val_idx] = val_prob

    best_t, best_f1 = find_optimal_threshold(y_val, val_prob)
    fold_f1s.append(best_f1)
    print(f'Fold {fold+1} → threshold={best_t:.3f}  F1={best_f1:.5f}')

    test_preds += model.predict_proba(X_te[fold_features])[:, 1] / N_FOLDS
    last_model, last_fold_features = model, fold_features

print(f'\n══════════════════════════════════════════')
print(f'CV F1 scores: {[f"{f:.4f}" for f in fold_f1s]}')
print(f'Mean CV F1 : {np.mean(fold_f1s):.5f} ± {np.std(fold_f1s):.5f}')

## 5. Optimal Global Threshold

We search for the best decision threshold on the **full OOF probabilities** — this gives a more stable estimate than per-fold thresholds.

In [ ]:
global_threshold, global_f1 = find_optimal_threshold(y, oof_preds)
print(f'Global OOF F1: {global_f1:.5f}  at threshold={global_threshold:.3f}')

print('\nThreshold sweep:')
for t in np.arange(global_threshold - 0.05, global_threshold + 0.06, 0.01):
    f = f1_score(y, (oof_preds >= t).astype(int))
    marker = '  ← best' if abs(t - global_threshold) < 0.006 else ''
    print(f'  t={t:.3f}  F1={f:.5f}{marker}')

## 6. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature':    last_fold_features,
    'importance': last_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 20 features:')
print(importance_df.head(20).to_string(index=False))

## 7. Generate Submission

In [ ]:
final_preds = (test_preds >= global_threshold).astype(bool)

sub['isSold'] = final_preds
sub.to_csv('submission_lgbm.csv', index=False)

print(f'submission_lgbm.csv saved')
print(f'Predicted positive rate: {final_preds.mean():.4f}')
print(f'  True: {final_preds.sum():,}  |  False: {(~final_preds).sum():,}')
sub.head()

## 8. What to Try Next (to push toward 0.777+)

| Idea | Expected gain |
|---|---|
| **CatBoost or XGBoost ensemble** | +0.005–0.010 |
| **Interaction features** (`country×device`, `placement×hour`) | +0.005 |
| **Multi-level target encoding** (e.g. `websiteId×hour`) | +0.005 |
| **Pseudo-labeling** on high-confidence test predictions | +0.003 |
| **Optuna hyperparameter search** | +0.003 |
| **More `num_leaves` + lower `learning_rate`** | +0.002 |